# 📈 Modelos Econométricos — Trabajo Final de Grado
## Riesgo País Argentino: Regresiones de Series de Tiempo

**Autor:** Martín  
**Universidad:** Universidad Nacional de Córdoba

---

Este notebook permite:
1. Seleccionar variables del catálogo y definir la variable dependiente
2. Configurar **múltiples modelos simultáneamente** (ARDL, VAR, VECM)
3. Definir **lags por serie**, horizonte temporal y parte del modelo donde actúa cada variable
4. Ejecutar **tests previos** (estacionariedad ADF/KPSS, cointegración de Johansen, causalidad de Granger)
5. Estimar todos los modelos configurados y comparar resultados

## 1. Configuración y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
import os
import sys
from datetime import datetime

sys.path.append(os.path.abspath('..'))
from src.models import estimar_ols, estimar_var, estimar_vecm, estimar_ardl, comparar_modelos
from src.utils import (
    test_estacionariedad, test_causalidad_granger,
    transformar_serie, crear_rezagos,
    resumen_estadistico, metricas_prediccion
)
from src.glosario import GLOSARIO, cargar_serie, buscar, get, info

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Módulos cargados correctamente.')

## 2. Catálogo de Variables Disponibles

Cargar todas las series disponibles desde el glosario a un DataFrame unificado.

In [ ]:
# Construir DataFrame consolidado con todas las series disponibles del glosario
series_disponibles = []
errores_carga = []

for entry in GLOSARIO:
    try:
        serie = cargar_serie(entry['id'])
        if serie is not None and len(serie) > 0:
            series_disponibles.append(serie)
    except Exception as e:
        errores_carga.append((entry['id'], str(e)))

if series_disponibles:
    df_todas = pd.concat(series_disponibles, axis=1)
    print(f'✅ Cargadas {len(df_todas.columns)} variables')
    print(f'   Período: {df_todas.index.min().date()} → {df_todas.index.max().date()}')
    if errores_carga:
        print(f'   ⚠️ {len(errores_carga)} series con error de carga')
else:
    df_todas = pd.DataFrame()
    print('❌ No se pudo cargar ninguna serie. Verificar rutas de archivos.')

# Crear diccionario de metadatos: id → {nombre, frecuencia, categoria, unidad}
metadatos = {}
for entry in GLOSARIO:
    metadatos[entry['id']] = {
        'nombre': entry['nombre'],
        'frecuencia': entry.get('frecuencia', 'diaria'),
        'categoria': entry.get('categoria', 'N/A'),
        'unidad': entry.get('unidad', 'N/A'),
        'descripcion': entry.get('descripcion', ''),
    }

# Construir lista de opciones para dropdowns: (label, id)
opciones_variables = []
if not df_todas.empty:
    for col in df_todas.columns:
        meta = metadatos.get(col, {})
        freq = meta.get('frecuencia', '')[0].upper() if meta.get('frecuencia') else '?'
        opciones_variables.append((f'[{freq}] {meta.get("nombre", col)} ({col})', col))

print(f'\nVariables en el catálogo: {len(GLOSARIO)}')
print(f'Variables cargadas exitosamente: {len(opciones_variables)}')
display(pd.DataFrame([
    {'ID': e['id'], 'Nombre': e['nombre'], 'Frecuencia': e.get('frecuencia', 'N/A'),
     'Categoría': e.get('categoria', 'N/A'), 'Disponible': '✅' if e.get('disponible', True) else '❌'}
    for e in GLOSARIO
]).set_index('ID').head(20))
print('... (ver catálogo completo con buscar())')

## 3. Configuración Interactiva de Modelos

Usar los widgets para seleccionar variables, frecuencia, tipo de modelo y sus parámetros.
Podés configurar **más de un modelo a la vez**.

In [ ]:
# ============================================================
# WIDGETS DE SELECCIÓN GLOBAL
# ============================================================

style_desc = {'description_width': '180px'}
layout_w = widgets.Layout(width='420px')
layout_wide = widgets.Layout(width='600px')

# --- Variable dependiente ---
wdep_dropdown = widgets.Dropdown(
    options=opciones_variables if opciones_variables else [('Sin variables cargadas', '')],
    value=opciones_variables[0][1] if opciones_variables else '',
    description='Var. Dependiente (Y):',
    style=style_desc,
    layout=layout_wide
)

# --- Frecuencia ---
wfreq_dropdown = widgets.Dropdown(
    options=[('Diaria (D)', 'D'), ('Semanal (W)', 'W'), ('Mensual (M)', 'M'), ('Trimestral (Q)', 'Q')],
    value='M',
    description='Frecuencia:',
    style=style_desc,
    layout=layout_w
)

# --- Transformación ---
wtransf_dropdown = widgets.Dropdown(
    options=[('Nivel (sin transformar)', 'nivel'), ('Logaritmo natural', 'log'),
             ('Primera diferencia', 'diff'), ('Diff log (retornos log)', 'log_diff'),
             ('Variación porcentual', 'pct_change'), ('Z-score', 'z_score')],
    value='nivel',
    description='Transformación:',
    style=style_desc,
    layout=layout_w
)

# --- Fechas ---
wstart_text = widgets.Text(
    value='2005-01-01',
    description='Fecha inicio:',
    style=style_desc,
    layout=layout_w,
    placeholder='YYYY-MM-DD'
)
wend_text = widgets.Text(
    value='2024-12-31',
    description='Fecha fin:',
    style=style_desc,
    layout=layout_w,
    placeholder='YYYY-MM-DD'
)

# --- Cantidad de modelos ---
wnmodelos_slider = widgets.IntSlider(
    value=1, min=1, max=5, step=1,
    description='N° de modelos:',
    style=style_desc,
    layout=layout_w
)

print('✅ Widgets creados. Ejecutar la siguiente celda para ver la interfaz completa.')

In [ ]:
# ============================================================
# INTERFAZ COMPLETA: CONFIGURACIÓN DE MODELOS
# ============================================================

modelo_widgets = []  # Guarda referencias a los widgets de cada modelo
modelo_accordion = widgets.Accordion()
current_n_models = wnmodelos_slider.value

def crear_panel_modelo(idx):
    """Crea los widgets de configuración para un modelo individual."""
    panel = {}
    
    panel['tipo'] = widgets.Dropdown(
        options=[('ARDL', 'ARDL'), ('VAR', 'VAR'), ('VECM', 'VECM'), ('OLS (comparación)', 'OLS')],
        value='ARDL',
        description=f'M{idx} — Tipo:',
        style=style_desc,
        layout=layout_w
    )
    
    panel['vars_ind'] = widgets.SelectMultiple(
        options=opciones_variables if opciones_variables else [('Sin variables', '')],
        value=(),
        description=f'M{idx} — Vars. Ind.:',
        style=style_desc,
        layout=widgets.Layout(width='600px', height='150px')
    )
    
    panel['max_lags_dep'] = widgets.IntSlider(
        value=4, min=0, max=12, step=1,
        description='Lags Y (dependiente):',
        style=style_desc,
        layout=layout_w
    )
    
    panel['max_lags_ind'] = widgets.IntSlider(
        value=4, min=0, max=12, step=1,
        description='Lags X (independientes):',
        style=style_desc,
        layout=layout_w
    )
    
    panel['k_ar_diff'] = widgets.IntSlider(
        value=1, min=0, max=8, step=1,
        description='k_ar_diff (VECM):',
        style=style_desc,
        layout=layout_w
    )
    
    panel['det_order'] = widgets.Dropdown(
        options=[('Sin determinístico (-1)', -1), ('Constante restringida (0)', 0),
                 ('Constante no restringida (1)', 1)],
        value=0,
        description='Det. order (VECM):',
        style=style_desc,
        layout=layout_w
    )
    
    panel['criterio'] = widgets.Dropdown(
        options=[('AIC', 'aic'), ('BIC', 'bic'), ('HQIC', 'hqic')],
        value='aic',
        description='Criterio selección:',
        style=style_desc,
        layout=layout_w
    )
    
    # Etiqueta para el nombre del modelo
    panel['nombre'] = widgets.Text(
        value=f'Modelo {idx}',
        description='Nombre:',
        style=style_desc,
        layout=layout_w
    )
    
    vbox = widgets.VBox([
        panel['nombre'],
        panel['tipo'],
        panel['vars_ind'],
        widgets.HBox([panel['max_lags_dep'], panel['max_lags_ind']]),
        panel['criterio'],
        widgets.HBox([panel['k_ar_diff'], panel['det_order']]),
    ])
    
    return panel, vbox


def actualizar_interfaz(change=None):
    """Reconstruye los paneles de modelo cuando cambia el número de modelos."""
    global modelo_widgets, modelo_accordion, current_n_models
    n = wnmodelos_slider.value
    if n == current_n_models and modelo_widgets:
        return
    current_n_models = n
    modelo_widgets = []
    children = []
    titles = []
    for i in range(1, n + 1):
        panel, vbox = crear_panel_modelo(i)
        modelo_widgets.append(panel)
        children.append(vbox)
        titles.append(f'Modelo {i}')
    modelo_accordion.children = children
    for i, title in enumerate(titles):
        modelo_accordion.set_title(i, title)


wnmodelos_slider.observe(actualizar_interfaz, names='value')
actualizar_interfaz()

# Layout global
global_config = widgets.VBox([
    widgets.HTML('<h3 style="margin-top:0;">⚙️ Configuración Global</h3>'),
    wdep_dropdown,
    widgets.HBox([wfreq_dropdown, wtransf_dropdown]),
    widgets.HBox([wstart_text, wend_text]),
    wnmodelos_slider,
])

display(global_config)
print('\n📋 Configuración por Modelo (expandir cada pestaña):')
display(modelo_accordion)

## 4. Tests Previos

Seleccionar los tests a ejecutar **antes** de estimar los modelos.

In [ ]:
# Widgets para tests previos
wtest_adf = widgets.Checkbox(value=True, description='Test ADF (estacionariedad)', style=style_desc)
wtest_kpss = widgets.Checkbox(value=True, description='Test KPSS (estacionariedad)', style=style_desc)
wtest_granger = widgets.Checkbox(value=True, description='Causalidad de Granger', style=style_desc)
wtest_johansen = widgets.Checkbox(value=True, description='Cointegración de Johansen (VECM)', style=style_desc)
wtest_corr = widgets.Checkbox(value=False, description='Matriz de correlaciones', style=style_desc)

tests_box = widgets.VBox([
    widgets.HTML('<h3>📋 Seleccionar Tests Previos</h3>'),
    wtest_adf, wtest_kpss, wtest_granger, wtest_johansen, wtest_corr
])

btn_preparar = widgets.Button(
    description='▶ Preparar Datos y Ejecutar Tests',
    button_style='primary',
    layout=widgets.Layout(width='320px')
)

output_pre = widgets.Output()

display(tests_box, btn_preparar, output_pre)

In [ ]:
# ============================================================
# FUNCIÓN: OBTENER CONFIGURACIÓN DE TODOS LOS MODELOS
# ============================================================

def obtener_config_modelos():
    """
    Lee los widgets y devuelve una lista de dicts con la configuración de cada modelo.
    """
    configs = []
    for i, panel in enumerate(modelo_widgets):
        tipo = panel['tipo'].value
        vars_ind = list(panel['vars_ind'].value)
        if not vars_ind and tipo != 'VAR':
            continue  # Saltar modelos sin variables
        config = {
            'nombre': panel['nombre'].value or f'Modelo {i+1}',
            'tipo': tipo,
            'vars_ind': vars_ind,
            'max_lags_dep': panel['max_lags_dep'].value,
            'max_lags_ind': panel['max_lags_ind'].value,
            'k_ar_diff': panel['k_ar_diff'].value,
            'det_order': panel['det_order'].value,
            'criterio': panel['criterio'].value,
        }
        configs.append(config)
    return configs


def preparar_dataframe():
    """
    Prepara el DataFrame de trabajo: frecuencia, transformación, rango de fechas.
    """
    var_dep = wdep_dropdown.value
    vars_ind_set = set()
    for panel in modelo_widgets:
        vars_ind_set.update(panel['vars_ind'].value)
    
    todas = [var_dep] + list(vars_ind_set) if var_dep else list(vars_ind_set)
    if not todas:
        return None, None, None
    
    # Filtrar columnas que existen
    cols_existentes = [c for c in todas if c in df_todas.columns]
    df = df_todas[cols_existentes].copy()
    
    # Resampleo
    freq_map = {'D': 'D', 'W': 'W', 'M': 'M', 'Q': 'Q'}
    freq = wfreq_dropdown.value
    if freq != 'D':
        df = df.resample(freq).last()
    
    # Transformación
    trans = wtransf_dropdown.value
    for col in df.columns:
        df[col] = transformar_serie(df[col], trans)
    
    # Filtro de fechas
    if wstart_text.value:
        try:
            df = df[pd.Timestamp(wstart_text.value):]
        except:
            pass
    if wend_text.value:
        try:
            df = df[:pd.Timestamp(wend_text.value)]
        except:
            pass
    
    df = df.dropna()
    return df, var_dep, cols_existentes

In [ ]:
# ============================================================
# EJECUCIÓN DE TESTS PREVIOS
# ============================================================

def ejecutar_tests_previos(b):
    with output_pre:
        clear_output(wait=True)
        
        df, var_dep, cols_usadas = preparar_dataframe()
        if df is None or df.empty:
            print('❌ No hay datos para testear. Verificar variables y fechas.')
            return
        
        configs = obtener_config_modelos()
        print(f'📊 Datos preparados: {df.shape[0]} obs × {df.shape[1]} vars')
        print(f'   Período: {df.index.min().date()} → {df.index.max().date()}')
        print(f'   Frecuencia: {wfreq_dropdown.value} | Transformación: {wtransf_dropdown.value}')
        print(f'   Var. dependiente: {var_dep}')

        # ----- ESTACIONARIEDAD -----
        if wtest_adf.value or wtest_kpss.value:
            print(f'\n{"="*70}')
            print('📋 TESTS DE ESTACIONARIEDAD (ADF + KPSS)')
            print(f'{"="*70}')
            resultados_est = []
            for col in df.columns:
                res = test_estacionariedad(df[col], col)
                resultados_est.append(res)
            df_est = pd.DataFrame(resultados_est)
            display(df_est[['Variable', 'ADF_pvalue', 'KPSS_pvalue', 'Conclusión']])
            
            # Guardar en variable global para uso posterior
            global _df_estacionariedad
            _df_estacionariedad = df_est

        # ----- CAUSALIDAD GRANGER -----
        vars_ind_todas = set()
        for cfg in configs:
            vars_ind_todas.update(cfg['vars_ind'])
        
        if wtest_granger.value and var_dep and vars_ind_todas:
            print(f'\n{"="*70}')
            print('📋 TESTS DE CAUSALIDAD DE GRANGER')
            print(f'{"="*70}')
            resultados_gr = []
            for var in sorted(vars_ind_todas):
                if var in df.columns:
                    try:
                        res = test_causalidad_granger(df, var_dep, var, max_lags=12)
                        causal = '✅ Sí' if res['Causalidad_Granger'] else '❌ No'
                        resultados_gr.append({
                            'Variable': res['Variable_Ind'],
                            'Mejor Lag': res['Mejor_Lag'],
                            'P-Value': res['P_Value'],
                            'Causa Granger a Y': causal
                        })
                    except Exception as e:
                        print(f'   ⚠️ {var}: error — {e}')
            if resultados_gr:
                display(pd.DataFrame(resultados_gr).set_index('Variable'))
            else:
                print('   No se pudo calcular causalidad.')

        # ----- COINTEGRACIÓN JOHANSEN (para modelos VECM) -----
        modelos_vecm = [c for c in configs if c['tipo'] == 'VECM']
        if wtest_johansen.value and modelos_vecm:
            from statsmodels.tsa.vector_ar.vecm import coint_johansen
            print(f'\n{"="*70}')
            print('📋 TEST DE COINTEGRACIÓN DE JOHANSEN (VECM)')
            print(f'{"="*70}')
            for cfg in modelos_vecm:
                vars_modelo = [var_dep] + [v for v in cfg['vars_ind'] if v in df.columns]
                if len(vars_modelo) < 2:
                    continue
                data_coint = df[vars_modelo].dropna()
                if data_coint.shape[1] < 2:
                    continue
                try:
                    joh = coint_johansen(data_coint, det_order=cfg['det_order'], k_ar_diff=cfg['k_ar_diff'])
                    print(f'\n--- {cfg["nombre"]} ---')
                    print(f'   Variables: {", ".join(vars_modelo)}')
                    print(f'\n   Hipótesis: r=0 → al menos 1 relación de cointegración')
                    for r in range(len(joh.lr1)):
                        signif_trace = '✅' if joh.lr1[r] > joh.cvt[r, 1] else '  '
                        signif_max = '✅' if joh.lr2[r] > joh.cvm[r, 1] else '  '
                        print(f'   r ≤ {r}: Trace={joh.lr1[r]:.2f} (cv 5%={joh.cvt[r,1]:.2f}) {signif_trace}  |  '
                              f'Max-Eigen={joh.lr2[r]:.2f} (cv 5%={joh.cvm[r,1]:.2f}) {signif_max}')
                    n_coint = sum(joh.lr1 > joh.cvt[:, 1])
                    print(f'\n   → Relaciones de cointegración (Trace, 5%): {n_coint}')
                except Exception as e:
                    print(f'   ❌ Error en {cfg["nombre"]}: {e}')

        # ----- MATRIZ DE CORRELACIONES -----
        if wtest_corr.value:
            print(f'\n{"="*70}')
            print('📋 MATRIZ DE CORRELACIONES')
            print(f'{"="*70}')
            vars_corr = [var_dep] + sorted(vars_ind_todas) if var_dep else sorted(vars_ind_todas)
            vars_corr = [v for v in vars_corr if v in df.columns]
            if vars_corr:
                corr_mat = df[vars_corr].corr()
                display(corr_mat.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}'))

        print(f'\n✅ Tests previos completados.')

# Conectar botón
btn_preparar.on_click(ejecutar_tests_previos)

## 5. Estimación de Modelos

Ejecutar todos los modelos configurados y ver resultados.

In [ ]:
btn_estimar = widgets.Button(
    description='🚀 ESTIMAR TODOS LOS MODELOS',
    button_style='success',
    layout=widgets.Layout(width='320px')
)

output_modelos = widgets.Output()

display(btn_estimar, output_modelos)

In [ ]:
# Almacenamiento global de resultados para comparación
_resultados_globales = {}
_df_trabajo = None
_var_dep_trabajo = None


def estimar_modelos(b):
    global _resultados_globales, _df_trabajo, _var_dep_trabajo
    
    with output_modelos:
        clear_output(wait=True)
        
        df, var_dep, _ = preparar_dataframe()
        if df is None or df.empty:
            print('❌ No hay datos. Ejecutar primero "Tests Previos" para verificar.')
            return
        
        _df_trabajo = df
        _var_dep_trabajo = var_dep
        
        configs = obtener_config_modelos()
        if not configs:
            print('❌ No hay modelos configurados. Agregar al menos un modelo en la Sección 3.')
            return
        
        _resultados_globales = {}
        
        for i, cfg in enumerate(configs):
            tipo = cfg['tipo']
            nombre = cfg['nombre']
            vars_ind = [v for v in cfg['vars_ind'] if v in df.columns]
            todas_vars = [var_dep] + vars_ind if var_dep and var_dep in df.columns else vars_ind
            
            print(f'\n{"="*70}')
            print(f'📊 {nombre} — {tipo}')
            print(f'{"="*70}')
            
            if tipo == 'ARDL':
                _estimar_ardl_modelo(df, var_dep, vars_ind, nombre, cfg)
            elif tipo == 'VAR':
                _estimar_var_modelo(df, todas_vars, nombre, cfg)
            elif tipo == 'VECM':
                _estimar_vecm_modelo(df, todas_vars, nombre, cfg)
            elif tipo == 'OLS':
                _estimar_ols_modelo(df, var_dep, vars_ind, nombre)
        
        # Comparación automática al final
        if len(_resultados_globales) > 1:
            print(f'\n{"="*70}')
            print('📊 COMPARACIÓN MULTIMODELO')
            print(f'{"="*70}')
            df_comp = comparar_modelos(_resultados_globales)
            display(df_comp)
            print(f'\n✅ {len(_resultados_globales)} modelos estimados.')
        elif len(_resultados_globales) == 1:
            print(f'\n✅ 1 modelo estimado.')


def _estimar_ardl_modelo(df, var_dep, vars_ind, nombre, cfg):
    try:
        res = estimar_ardl(
            df, var_dep, vars_ind,
            max_lags_dep=cfg['max_lags_dep'],
            max_lags_ind=cfg['max_lags_ind'],
            criterio=cfg['criterio']
        )
        print(f'   Orden ARDL seleccionado: {res["orden_seleccionado"]}')
        print(f'   AR Lags: {res["ar_lags"]}')
        print(res['resumen'])
        _resultados_globales[nombre] = {
            'aic': res['modelo'].aic,
            'bic': res['modelo'].bic,
            'orden': res['orden_seleccionado'],
            'modelo_obj': res['modelo'],
        }
        # Gráfico de residuos
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                           subplot_titles=['Ajuste vs Real', 'Residuos'])
        fig.add_trace(go.Scatter(x=res['modelo'].fittedvalues.index, y=res['modelo'].fittedvalues,
                                name='Ajustado', line=dict(color='red')), row=1, col=1)
        y_vals = res['modelo'].model.endog
        fig.add_trace(go.Scatter(x=df.index[-len(y_vals):], y=y_vals,
                                name='Real', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=res['modelo'].fittedvalues.index, y=res['modelo'].resid,
                                name='Residuos', line=dict(color='gray')), row=2, col=1)
        fig.update_layout(height=450, template='plotly_white', title=f'ARDL — {nombre}')
        fig.show()
    except Exception as e:
        print(f'   ❌ Error en ARDL: {e}')
        import traceback
        traceback.print_exc()


def _estimar_var_modelo(df, todas_vars, nombre, cfg):
    try:
        res = estimar_var(df, todas_vars, max_lags=cfg['max_lags_dep'], criterio=cfg['criterio'])
        print(f'   Rezagos óptimos ({cfg["criterio"].upper()}): {res["lags_optimos"]}')
        print(f'   AIC: {res["modelo"].aic:.2f} | BIC: {res["modelo"].bic:.2f}')
        _resultados_globales[nombre] = {
            'aic': res['modelo'].aic,
            'bic': res['modelo'].bic,
            'lags_optimos': res['lags_optimos'],
            'modelo_obj': res['modelo'],
        }
        # IRF plot
        fig = res['irf'].plot(orth=True)
        fig.set_size_inches(14, 10)
        plt.tight_layout()
        plt.suptitle(f'Funciones Impulso-Respuesta — {nombre}', fontsize=14, y=1.01)
        plt.show()
    except Exception as e:
        print(f'   ❌ Error en VAR: {e}')


def _estimar_vecm_modelo(df, todas_vars, nombre, cfg):
    try:
        from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
        data = df[todas_vars].dropna()
        
        # Paso 1: Johansen
        joh = coint_johansen(data, det_order=cfg['det_order'], k_ar_diff=cfg['k_ar_diff'])
        n_coint = sum(joh.lr1 > joh.cvt[:, 1])
        print(f'   Test de Johansen: {n_coint} relaciones de cointegración significativas (5%)')
        
        if n_coint == 0:
            print('   ⚠️  No se detectó cointegración. El VECM no es apropiado.')
            print('   → Considerar estimar un VAR en diferencias en su lugar.')
            return
        
        # Paso 2: Estimar VECM
        vecm = VECM(data, k_ar_diff=cfg['k_ar_diff'], coint_rank=n_coint, deterministic='ci')
        resultado = vecm.fit()
        
        print(f'   Cointegration rank: {n_coint}')
        print(f'   AIC: {resultado.aic:.2f} | BIC: {resultado.bic:.2f}')
        print(resultado.summary())
        
        _resultados_globales[nombre] = {
            'aic': resultado.aic,
            'bic': resultado.bic,
            'n_cointegracion': n_coint,
            'modelo_obj': resultado,
        }
    except Exception as e:
        print(f'   ❌ Error en VECM: {e}')


def _estimar_ols_modelo(df, var_dep, vars_ind, nombre):
    try:
        res = estimar_ols(df, var_dep, vars_ind, hac=True)
        print(f'   R²: {res["r2"]:.4f} | R² Adj: {res["r2_adj"]:.4f}')
        print(f'   AIC: {res["aic"]:.2f} | BIC: {res["bic"]:.2f}')
        print(f'   Durbin-Watson: {res["durbin_watson"]:.4f}')
        print(res['resumen'])
        _resultados_globales[nombre] = {
            'aic': res['aic'],
            'bic': res['bic'],
            'r2': res['r2'],
            'r2_adj': res['r2_adj'],
            'modelo_obj': res['modelo'],
        }
    except Exception as e:
        print(f'   ❌ Error en OLS: {e}')


btn_estimar.on_click(estimar_modelos)

## 6. Diagnósticos de Residuos del Último Modelo Estimado

In [ ]:
def mostrar_diagnosticos():
    """Muestra diagnósticos de residuos para el último modelo estimado que tenga método resid."""
    if not _resultados_globales:
        print('⚠️ Ejecutar primero la estimación de modelos.')
        return
    
    import statsmodels.api as sm
    
    for nombre, res in _resultados_globales.items():
        modelo = res.get('modelo_obj')
        if modelo is None:
            continue
        
        # Intentar obtener residuos
        try:
            if hasattr(modelo, 'resid'):
                residuos = modelo.resid
                fitted = modelo.fittedvalues
            elif hasattr(modelo, 'results'):
                residuos = modelo.results.resid
                fitted = modelo.results.fittedvalues
            else:
                continue
        except:
            continue
        
        print(f'\n📊 Diagnósticos — {nombre}')
        print('='*60)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        axes[0, 0].scatter(fitted, residuos, alpha=0.5, s=10)
        axes[0, 0].axhline(y=0, color='r', linestyle='--')
        axes[0, 0].set_title('Residuos vs Valores Ajustados')
        
        sm.qqplot(residuos.dropna() if hasattr(residuos, 'dropna') else residuos,
                  line='45', ax=axes[0, 1])
        axes[0, 1].set_title('QQ Plot de Residuos')
        
        axes[1, 0].hist(residuos, bins=30, edgecolor='black', alpha=0.7)
        axes[1, 0].set_title('Distribución de Residuos')
        
        sm.graphics.tsa.plot_acf(residuos.dropna() if hasattr(residuos, 'dropna') else residuos,
                                 lags=20, ax=axes[1, 1])
        axes[1, 1].set_title('ACF de Residuos')
        
        plt.suptitle(f'Diagnósticos — {nombre}', fontsize=14)
        plt.tight_layout()
        plt.show()

mostrar_diagnosticos()

## 7. Validación Out-of-Sample

Dividir los datos en train/test y evaluar capacidad predictiva.

In [ ]:
# Widgets para out-of-sample
wpctaje_slider = widgets.FloatSlider(
    value=0.20, min=0.10, max=0.40, step=0.05,
    description='% Test:',
    style=style_desc,
    layout=layout_w
)

btn_oos = widgets.Button(
    description='📈 Validación Out-of-Sample',
    button_style='warning',
    layout=widgets.Layout(width='280px')
)

output_oos = widgets.Output()

display(widgets.HBox([wpctaje_slider, btn_oos]), output_oos)

In [ ]:
def validar_oos(b):
    with output_oos:
        clear_output(wait=True)
        
        if _df_trabajo is None or _var_dep_trabajo is None:
            print('❌ Ejecutar primero la estimación de modelos (Sección 5).')
            return
        
        df = _df_trabajo.copy()
        var_dep = _var_dep_trabajo
        configs = obtener_config_modelos()
        
        n = len(df)
        n_test = int(n * wpctaje_slider.value)
        n_train = n - n_test
        df_train = df.iloc[:n_train]
        df_test = df.iloc[n_train:]
        
        print(f'📊 Split Train/Test')
        print(f'   Train: {len(df_train)} obs ({df_train.index.min().date()} → {df_train.index.max().date()})')
        print(f'   Test:  {len(df_test)} obs ({df_test.index.min().date()} → {df_test.index.max().date()})')
        
        for cfg in configs:
            tipo = cfg['tipo']
            nombre = cfg['nombre']
            vars_ind = [v for v in cfg['vars_ind'] if v in df.columns]
            
            if tipo != 'ARDL' or not vars_ind:
                continue
            
            print(f'\n--- {nombre} ({tipo}) ---')
            
            try:
                from statsmodels.tsa.ardl import ARDL, ardl_select_order
                
                y_train = df_train[var_dep]
                X_train = df_train[vars_ind]
                
                sel = ardl_select_order(
                    y_train, cfg['max_lags_dep'],
                    X_train, cfg['max_lags_ind'],
                    ic=cfg['criterio'], trend='c'
                )
                modelo_train = sel.model.fit()
                
                # Predicción dinámica en test
                y_pred = modelo_train.predict(
                    exog=df_test[vars_ind],
                    start=df_test.index[0],
                    end=df_test.index[-1],
                    dynamic=True
                )
                y_real = df_test[var_dep]
                
                # Alinear índices
                common_idx = y_real.index.intersection(y_pred.index)
                y_real_al = y_real.loc[common_idx]
                y_pred_al = y_pred.loc[common_idx]
                
                if len(y_real_al) > 0:
                    metricas = metricas_prediccion(y_real_al.values, y_pred_al.values)
                    for k, v in metricas.items():
                        print(f'   {k}: {v}')
                    
                    fig = go.Figure()
                    fig.add_trace(go.Scatter(x=y_real_al.index, y=y_real_al, name='Real', line=dict(color='blue')))
                    fig.add_trace(go.Scatter(x=y_pred_al.index, y=y_pred_al, name='Predicción',
                                            line=dict(color='red', dash='dash')))
                    fig.update_layout(title=f'Out-of-Sample — {nombre}', template='plotly_white', height=400)
                    fig.show()
                else:
                    print('   ⚠️ Sin datos comunes para comparar.')
            except Exception as e:
                print(f'   ❌ Error: {e}')

btn_oos.on_click(validar_oos)

## 8. Búsqueda Rápida en el Catálogo

Filtrar variables por texto, fuente o categoría.

In [ ]:
wbuscar_text = widgets.Text(
    value='riesgo',
    description='Buscar:',
    style=style_desc,
    layout=widgets.Layout(width='400px'),
    placeholder='Ej: reservas, deuda, pbi...'
)

wbuscar_fuente = widgets.Dropdown(
    options=[('Todas', ''), ('BCRA', 'bcra'), ('FRED', 'fred'), ('Global', 'global'),
             ('INDEC', 'indec'), ('MECON', 'mecon'), ('World Bank', 'worldbank'),
             ('Processed', 'processed'), ('Local', 'local')],
    value='',
    description='Fuente:',
    style=style_desc,
    layout=layout_w
)

wbuscar_cat = widgets.Dropdown(
    options=[('Todas', ''), ('Dependiente', 'dependiente'), ('Mercado', 'mercado'),
             ('Cambiario', 'cambiario'), ('Monetario', 'monetario'), ('Real', 'real'),
             ('Externo', 'externo'), ('Fiscal', 'fiscal'), ('Commodity', 'commodity'),
             ('Institucional', 'institucional'), ('Dummy', 'dummy')],
    value='',
    description='Categoría:',
    style=style_desc,
    layout=layout_w
)

btn_buscar = widgets.Button(description='🔍 Buscar', button_style='info')
output_buscar = widgets.Output()

display(wbuscar_text, widgets.HBox([wbuscar_fuente, wbuscar_cat, btn_buscar]), output_buscar)


def ejecutar_busqueda(b):
    with output_buscar:
        clear_output(wait=True)
        df_result = buscar(
            texto=wbuscar_text.value,
            fuente=wbuscar_fuente.value if wbuscar_fuente.value else '',
            categoria=wbuscar_cat.value if wbuscar_cat.value else ''
        )
        print(f'Resultados: {len(df_result)} series encontradas')
        display(df_result)

btn_buscar.on_click(ejecutar_busqueda)
# Ejecutar búsqueda inicial
ejecutar_busqueda(None)

---
### 📝 Resumen de uso

1. **Sección 2**: Carga automática de todas las variables desde el glosario.
2. **Sección 3**: Seleccionar variable dependiente, frecuencia, transformación, fechas y modelos.
   - Expandir cada pestaña de modelo para configurar tipo, variables y lags.
3. **Sección 4**: Ejecutar tests previos (estacionariedad, cointegración, causalidad).
4. **Sección 5**: Estimar todos los modelos configurados y ver resultados.
5. **Sección 6**: Diagnósticos de residuos (QQ, ACF, histograma).
6. **Sección 7**: Validación out-of-sample con split train/test.
7. **Sección 8**: Búsqueda rápida de variables en el catálogo.

**Tip**: Para modelos VECM, ejecutar primero los tests de cointegración (Sección 4). Si no hay cointegración, el modelo no es apropiado.